---

# Tugas Besar 2 IF3170 Inteligensi Artifisial

*This notebook is the assignment template. Each group works on its own copy, and cells or sections may be added as needed.*

---

## Group 07

- Name (NIM)
- Name (NIM)
- Name (NIM)
- Name (NIM)
- Name (NIM)

---

## Table of Contents

1. [**Introduction**](#1)
2. [**Initialization**](#2)
3. [**Data Understanding (Optional)**](#3)
4. [**Training and Validation Split**](#4)
5. [**Data Preparation**](#5)
6. [**Preparation Pipeline**](#6)
7. [**Modeling and Validation**](#7)
8. [**Comparison with scikit-learn**](#8)
9. [**Saving and Loading Models**](#9)
10. [**Kaggle Submission**](#10)
11. [**Error Analysis**](#11)


---

# Introduction <a name="1"></a>

---

This section frames the problem in the words of the group.

## Overview

*To be written by the group.*

## Aim

*To be written by the group.*

## Metric

*To be written by the group.*

## Dataset

*To be written by the group.*

## Approach

*To be written by the group.*

---

# Initialization <a name="2"></a>

---

This section prepares the environment, the settings, and the data files.

## Environment Setup

The following cell lists the base libraries. The install line is commented by default.

In [ ]:
# %pip install -q numpy pandas

: 

## Import Libraries

Every import of the notebook is kept in this cell, so does the custom model.

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
from src.svm import SVM
import sklearn as sk
import pickle
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.animation import FuncAnimation, FFMpegWriter, PillowWriter
# Preprocessing
from src.preprocessing import (
    DEFAULT_PREP_PARAMS,
    TARGET,
    Preprocessor,
    prepare_datasets,
    random_oversample,
    random_undersample,
    remove_duplicates,
    smote,
    stratified_split,
)

# Untuk modelnya
from sklearn.linear_model import LogisticRegression as SkLogisticRegression
from src.logistic_regression import LogisticRegression

## Settings

Paths and shared constants are kept in this class.

In [ ]:
class Settings:
    SEED       = 42
    DATA_DIR   = Path('data')
    OUTPUT_DIR = Path('output')

CFG = Settings()
CFG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Seed Everything

A fixed seed keeps the results reproducible.

In [ ]:
def seed_everything(seed: int):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(CFG.SEED)

## Load Dataset

The competition files are read here. The `id` column stays as text because some values resemble numbers.

In [ ]:
train_df = pd.read_csv(CFG.DATA_DIR / 'train.csv', dtype={'id': str})
test_df  = pd.read_csv(CFG.DATA_DIR / 'test.csv', dtype={'id': str})

print(f'Train: {train_df.shape}  Test: {test_df.shape}')

---

# Data Understanding (Optional) <a name="3"></a>

---

Data understanding builds the knowledge that later preparation decisions rest on.

## Describe Data

This part covers the size, the attribute types, and the meaning of each column.

In [ ]:
CAT_COLS = ['meal_package', 'room_category', 'booking_channel']
FEATURE_COLS = [c for c in train_df.columns if c not in ('id', TARGET)]

display(train_df.head())
train_df.info()

overview = pd.DataFrame({
    'dtype': train_df.dtypes.astype(str),
    'n_null': train_df.isna().sum(),
    'null_%': (train_df.isna().mean() * 100).round(2),
    'n_unique': train_df.nunique(),
    'min': train_df.min(numeric_only=True),
    'max': train_df.max(numeric_only=True),
})
display(overview)
display(train_df.describe().T)

## Explore Data

This part covers distributions and relationships between attributes.

### Target Distribution

In [ ]:
counts = train_df[TARGET].value_counts().sort_index()
display(pd.DataFrame({'jumlah': counts, 'proporsi': (counts / counts.sum()).round(4)}))
print(f'rasio kelas 0 : kelas 1 = {counts[0] / counts[1]:.2f} : 1')

fig, ax = plt.subplots(figsize=(4.5, 3.5))
sns.barplot(x=counts.index.map({0: 'tidak batal (0)', 1: 'batal (1)'}), y=counts.values, ax=ax)
for i, v in enumerate(counts.values):
    ax.text(i, v, f'{v:,}\n({v / counts.sum():.1%})', ha='center', va='bottom')
ax.set_ylim(0, counts.max() * 1.18); ax.set_ylabel('jumlah'); ax.set_title('Distribusi target is_canceled')
fig.tight_layout(); fig.savefig(CFG.OUTPUT_DIR / 'eda_target_distribution.png', dpi=150)
plt.show()

### Feature Analysis

#### Distribusi fitur numerik menurut kelas

In [ ]:
NUM_PLOT = ['days_before_arrival', 'nightly_rate_idr', 'weekday_nights', 'weekend_nights',
            'prior_completed_bookings', 'prior_cancellations', 'num_children', 'special_requests']
fig, axes = plt.subplots(2, 4, figsize=(19, 8))
for ax, c in zip(axes.ravel(), NUM_PLOT):
    sns.histplot(data=train_df, x=c, hue=TARGET, stat='density', common_norm=False, element='step',
                 bins=30, ax=ax, palette={0: 'tab:blue', 1: 'tab:orange'})
    ax.set_title(c)
fig.tight_layout(); fig.savefig(CFG.OUTPUT_DIR / 'eda_numeric_distributions.png', dpi=150)
plt.show()

#### Rasio pembatalan menurut kategori

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for ax, c in zip(axes, CAT_COLS):
    g = train_df.groupby(c, dropna=False)[TARGET].agg(['mean', 'size']).sort_values('mean')
    g.index = g.index.fillna('(null)')
    sns.barplot(x=g['mean'], y=g.index.astype(str), ax=ax, color='tab:blue')
    for i, (m, n) in enumerate(zip(g['mean'], g['size'])):
        ax.text(m, i, f' {m:.2f} (n={n:,})', va='center', fontsize=8)
    ax.set_xlim(0, 0.75); ax.set_xlabel('rasio batal'); ax.set_title(c)
fig.tight_layout(); fig.savefig(CFG.OUTPUT_DIR / 'eda_categorical_cancel_rate.png', dpi=150)
plt.show()

#### Rasio pembatalan menurut fitur numerik

def rate_by(col, bins=None):
    key = train_df[col] if bins is None else pd.cut(train_df[col], bins)
    return train_df.groupby(key, observed=True)[TARGET].mean()

panels = {
    'days_before_arrival': ('days_before_arrival', [-1, 7, 30, 90, 180, 500]),
    'special_requests': ('special_requests', None),
    'arrival_year': ('arrival_year', None),
    'arrival_month': ('arrival_month', None),
    'is_returning_guest': ('is_returning_guest', None),
    'needs_parking': ('needs_parking', None),
}
fig, axes = plt.subplots(2, 3, figsize=(17, 8))
for ax, (title, (col, bins)) in zip(axes.ravel(), panels.items()):
    r = rate_by(col, bins)
    sns.barplot(x=r.index.astype(str), y=r.values, ax=ax, color='tab:blue')
    ax.set_title(f'Rasio batal per {title}'); ax.set_ylabel('rasio batal'); ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=30)
fig.tight_layout(); fig.savefig(CFG.OUTPUT_DIR / 'eda_numeric_cancel_rate.png', dpi=150)
plt.show()

#### Korelasi

In [ ]:
corr = train_df.corr(numeric_only=True)
fig, axes = plt.subplots(1, 2, figsize=(17, 6.5), gridspec_kw={'width_ratios': [1, 2.2]})
t = corr[TARGET].drop(TARGET).sort_values()
sns.heatmap(t.to_frame('korelasi dengan target'), annot=True, fmt='.2f', cmap='vlag', center=0, ax=axes[0], cbar=False)
sns.heatmap(corr, annot=True, fmt='.1f', cmap='vlag', center=0, ax=axes[1], annot_kws={'size': 6})
axes[0].set_title('Korelasi dengan target'); axes[1].set_title('Korelasi antar fitur numerik')
fig.tight_layout(); fig.savefig(CFG.OUTPUT_DIR / 'eda_correlation.png', dpi=150)
plt.show()

## Verify Data Quality

This part covers completeness, consistency, and the validity of values.

In [ ]:
null_table = pd.DataFrame({
    'train_null': train_df[FEATURE_COLS].isna().sum(),
    'train_%': (train_df[FEATURE_COLS].isna().mean() * 100).round(2),
    'test_null': test_df[FEATURE_COLS].isna().sum(),
    'test_%': (test_df[FEATURE_COLS].isna().mean() * 100).round(2),
})
display(null_table[null_table.train_null > 0])

dup_cols_all = [c for c in train_df.columns if c != 'id']
arrival_all = pd.to_datetime(dict(year=train_df.arrival_year, month=train_df.arrival_month, day=train_df.arrival_day),
                             errors='coerce')
checks = pd.Series({
    'id duplikat': int(train_df['id'].duplicated().sum()),
    'baris duplikat (semua kolom selain id)': int(train_df.duplicated(subset=dup_cols_all).sum()),
    'tanggal kedatangan tidak valid': int(arrival_all.isna().sum()),
    'total malam = 0 (train)': int(((train_df.weekend_nights + train_df.weekday_nights) == 0).sum()),
    'total malam = 0 (test)': int(((test_df.weekend_nights + test_df.weekday_nights) == 0).sum()),
    'num_adults = 0 (train)': int((train_df.num_adults == 0).sum()),
    'nightly_rate_idr = 0 (train)': int((train_df.nightly_rate_idr == 0).sum()),
    'num_children >= 9 (train)': int((train_df.num_children >= 9).sum()),
}, name='jumlah')
display(checks.to_frame())

---

# Training and Validation Split <a name="4"></a>

---

The split comes before data preparation so that validation data stays unseen. Only `train.csv` is split, and `test.csv` serves the Kaggle submission alone.

In [ ]:
VAL_SIZE = 0.2
train_set, val_set = stratified_split(train_df, target=TARGET, val_size=VAL_SIZE, seed=CFG.SEED)

split_table = pd.DataFrame({
    'baris': [len(train_set), len(val_set), len(test_df)],
    'proporsi kelas 1': [train_set[TARGET].mean(), val_set[TARGET].mean(), np.nan],
}, index=['train_set', 'val_set', 'test_df (Kaggle)']).round(4)
display(split_table)

---

# Data Preparation <a name="5"></a>

---

Each process below applies where the data calls for it, and the Insights block under it records the reason for the decision.

Preparation steps may use `sklearn`. Every statistic comes from the training split only, then transforms the other sets.

## Select Data

Selection decides which records and attributes stay in the analysis.

In [ ]:
ID_COL = 'id'
FEATURE_COLS = [c for c in train_set.columns if c not in (ID_COL, TARGET)]
print(f'{len(FEATURE_COLS)} fitur dipakai:')
print(FEATURE_COLS)

## Clean Data

Cleaning raises data quality to the level the chosen models need.

### Missing Values

This part covers missing values, if any.

In [ ]:
missing = pd.DataFrame({
    'train': train_set[FEATURE_COLS].isna().sum(),
    'val': val_set[FEATURE_COLS].isna().sum(),
    'test': test_df[FEATURE_COLS].isna().sum(),
})
missing = missing[missing.sum(axis=1) > 0]
missing['train_%'] = (missing.train / len(train_set) * 100).round(2)
display(missing)

rows = []
for c in missing.index:
    m = train_set[c].isna()
    rows.append((c, int(m.sum()), train_set.loc[m, TARGET].mean(), train_set.loc[~m, TARGET].mean()))
display(pd.DataFrame(rows, columns=['kolom', 'n_null', 'rasio batal | null', 'rasio batal | terisi']).round(3))

display(train_set.groupby('booking_channel')['nightly_rate_idr'].median().to_frame('median nightly_rate_idr'))

_imp = Preprocessor(**DEFAULT_PREP_PARAMS).fit(remove_duplicates(train_set))   # statistik dari train (tanpa duplikat, sama dengan pipeline akhir)
print('Median numerik (train):', {k: round(v, 2) for k, v in _imp.state_['median'].items() if k in missing.index})
print('Modus kategorikal (train):', _imp.state_['mode'])

### Duplicates

This part covers repeated records.

In [ ]:
dup_cols = [c for c in train_set.columns if c != 'id']
dup_mask = train_set.duplicated(subset=dup_cols, keep=False)
n_extra = int(train_set.duplicated(subset=dup_cols).sum())
label_conflict = int(train_set[dup_mask].groupby(dup_cols, dropna=False)[TARGET].nunique().gt(1).sum())
cross = val_set.merge(train_set.drop_duplicates(subset=dup_cols)[dup_cols], on=dup_cols, how='inner')

print(f'id duplikat                                   : {int(train_df.id.duplicated().sum())}')
print(f'baris duplikat berlebih di train_set          : {n_extra}')
print(f'kelompok duplikat dengan label bertentangan   : {label_conflict}')
print(f'baris val_set yang identik dengan baris train : {len(cross)}')

train_set = remove_duplicates(train_set, id_col='id')
print(f'train_set setelah menghapus duplikat          : {train_set.shape}')

### Outliers

This part covers values far from the rest of the data.

In [ ]:
OUTLIER_COLS = ['num_children', 'weekend_nights', 'weekday_nights', 'days_before_arrival',
                'prior_cancellations', 'prior_completed_bookings', 'nightly_rate_idr']
q = train_set[OUTLIER_COLS].quantile([0, 0.01, 0.25, 0.5, 0.75, 0.99, 1]).T
q.columns = ['min', 'p1', 'p25', 'p50', 'p75', 'p99', 'max']
display(q)

fig, axes = plt.subplots(1, len(OUTLIER_COLS), figsize=(19, 4))
for ax, c in zip(axes, OUTLIER_COLS):
    ax.boxplot(train_set[c].dropna()); ax.set_title(c, fontsize=9)
fig.tight_layout(); fig.savefig(CFG.OUTPUT_DIR / 'eda_outliers.png', dpi=150)
plt.show()

bounds = pd.DataFrame(_imp.state_['clip_bounds'], index=['batas bawah (p1)', 'batas atas (p99)']).T
bounds['train di luar batas'] = [
    int(((train_set[c] < lo) | (train_set[c] > hi)).sum()) if c in train_set else np.nan
    for c, (lo, hi) in _imp.state_['clip_bounds'].items()]
bounds['val di luar batas'] = [
    int(((val_set[c] < lo) | (val_set[c] > hi)).sum()) if c in val_set else np.nan
    for c, (lo, hi) in _imp.state_['clip_bounds'].items()]
print('Batas winsorizing (dari train; total_nights dan total_guests dihitung setelah fitur turunan dibuat):')
display(bounds)

fig, axes = plt.subplots(1, 3, figsize=(16, 3.8))
raw = train_set['nightly_rate_idr'].dropna()
lo, hi = _imp.state_['clip_bounds']['nightly_rate_idr']
sns.histplot(raw, bins=50, ax=axes[0]); axes[0].set_title('nightly_rate_idr mentah')
sns.histplot(raw.clip(lo, hi), bins=50, ax=axes[1]); axes[1].set_title('setelah winsorizing p1-p99')
sns.histplot(np.log1p(raw.clip(lo, hi)), bins=50, ax=axes[2]); axes[2].set_title('setelah winsorizing + log1p')
fig.tight_layout(); fig.savefig(CFG.OUTPUT_DIR / 'outlier_before_after.png', dpi=150)
plt.show()

### Data Errors

This part covers wrong types, inconsistent labels, and impossible values.

In [ ]:
nights_tr = train_set.weekend_nights + train_set.weekday_nights
arrival = pd.to_datetime(dict(year=train_set.arrival_year, month=train_set.arrival_month, day=train_set.arrival_day),
                         errors='coerce')
janggal = pd.DataFrame({
    'jumlah (train)': {
        'total malam = 0': int((nights_tr == 0).sum()),
        'num_adults = 0': int((train_set.num_adults == 0).sum()),
        'nightly_rate_idr = 0': int((train_set.nightly_rate_idr == 0).sum()),
        'num_children >= 9': int((train_set.num_children >= 9).sum()),
        'tanggal kedatangan tidak valid': int(arrival.isna().sum()),
        'nilai negatif (numerik)': int((train_set.select_dtypes('number') < 0).sum().sum()),
    },
    'jumlah (test)': {
        'total malam = 0': int(((test_df.weekend_nights + test_df.weekday_nights) == 0).sum()),
        'num_adults = 0': int((test_df.num_adults == 0).sum()),
        'nightly_rate_idr = 0': int((test_df.nightly_rate_idr == 0).sum()),
        'num_children >= 9': int((test_df.num_children >= 9).sum()),
        'tanggal kedatangan tidak valid': 0,
        'nilai negatif (numerik)': int((test_df.select_dtypes('number') < 0).sum().sum()),
    },
})
display(janggal)

print('Rasio batal keseluruhan:', round(train_set[TARGET].mean(), 3))
print('Total malam = 0 -> jumlah dan rasio batal:')
display(train_set.loc[nights_tr == 0, TARGET].agg(['size', 'mean']).to_frame('total malam = 0').T)
print('nightly_rate_idr = 0 per kanal -> jumlah dan rasio batal:')
display(train_set[train_set.nightly_rate_idr == 0].groupby('booking_channel')[TARGET].agg(['size', 'mean']))
print('Label kategori di test yang tidak ada di train:')
for c in CAT_COLS:
    print(f'  {c}:', set(test_df[c].dropna()) - set(train_set[c].dropna()) or 'tidak ada')

## Construct Data

Construction derives attributes and transforms values for modeling.

### Attribute Construction

<!-- rubric v3: derived columns, rate_per_guest_night -->

This part covers new attributes derived from existing ones.

In [ ]:
demo = train_set.copy()
demo['num_adults'] = demo.num_adults.fillna(_imp.state_['median']['num_adults'])
demo['num_children'] = demo.num_children.fillna(_imp.state_['median']['num_children'])
demo['total_nights'] = demo.weekend_nights + demo.weekday_nights
demo['total_guests'] = demo.num_adults + demo.num_children
demo['arrival_dow'] = pd.to_datetime(
    dict(year=demo.arrival_year, month=demo.arrival_month, day=demo.arrival_day)).dt.dayofweek
demo['arrival_month_sin'] = np.sin(2 * np.pi * (demo.arrival_month - 1) / 12)
demo['arrival_month_cos'] = np.cos(2 * np.pi * (demo.arrival_month - 1) / 12)
display(demo[['weekend_nights', 'weekday_nights', 'total_nights', 'num_adults', 'num_children', 'total_guests',
              'arrival_month', 'arrival_month_sin', 'arrival_month_cos', 'arrival_dow']].head())

fig, axes = plt.subplots(1, 3, figsize=(17, 4))
r = demo.groupby(demo.total_nights.clip(upper=8))[TARGET].mean()
sns.barplot(x=r.index.astype(str), y=r.values, ax=axes[0], color='tab:blue'); axes[0].set_title('Rasio batal per total_nights (8 = 8+)')
r = demo.groupby(demo.total_guests.clip(upper=5))[TARGET].mean()
sns.barplot(x=r.index.astype(str), y=r.values, ax=axes[1], color='tab:blue'); axes[1].set_title('Rasio batal per total_guests (5 = 5+)')
r = demo.groupby('arrival_dow')[TARGET].mean()
sns.barplot(x=r.index.astype(str), y=r.values, ax=axes[2], color='tab:blue'); axes[2].set_title('Rasio batal per hari kedatangan (0 = Senin)')
fig.tight_layout(); fig.savefig(CFG.OUTPUT_DIR / 'eda_constructed_features.png', dpi=150)
plt.show()

print('Korelasi |r| dengan target:')
display(demo[['total_nights', 'total_guests', 'arrival_dow', 'arrival_month_sin', 'arrival_month_cos',
              'weekend_nights', 'weekday_nights', 'num_adults', 'num_children', 'arrival_month']]
        .corrwith(demo[TARGET]).abs().sort_values(ascending=False).round(3).to_frame('|r|'))

### Feature Scaling

This part covers normalization to the range 0 to 1 and standardization.

In [ ]:
enc = Preprocessor(**DEFAULT_PREP_PARAMS).fit(train_set)   # statistik scaling dari train_set

num_cols = train_set.drop(columns=[TARGET]).select_dtypes('number').columns
before = train_set[num_cols].agg(['mean', 'std', 'min', 'max']).T
before.columns = ['mean (mentah)', 'std (mentah)', 'min (mentah)', 'max (mentah)']
display(before.sort_values('std (mentah)', ascending=False).round(3))

Z = enc.transform_frame(train_set)
num_feat = [c for c in Z.columns if '=' not in c]
after = Z[num_feat].agg(['mean', 'std', 'min', 'max']).T
after.columns = ['mean (z-score)', 'std (z-score)', 'min', 'max']
print('Setelah winsorizing, log1p, dan standardisasi z = (x - mean_train) / std_train:')
display(after.round(3))

fig, axes = plt.subplots(1, 2, figsize=(17, 5))
train_set[num_cols].plot.box(ax=axes[0], vert=False, logx=True); axes[0].set_title('Sebelum scaling (sumbu log)')
Z[num_feat].plot.box(ax=axes[1], vert=False); axes[1].set_title('Sesudah scaling')
fig.tight_layout(); fig.savefig(CFG.OUTPUT_DIR / 'scaling_before_after.png', dpi=150)
plt.show()

### Encoding

This part covers the numeric representation of categorical attributes.

In [ ]:
for c in CAT_COLS:
    display(train_set[c].value_counts(dropna=False).to_frame(c).T)

enc = Preprocessor(**DEFAULT_PREP_PARAMS).fit(train_set)
onehot_cols = [n for n in enc.feature_names_ if '=' in n]
print(f'{len(CAT_COLS)} kolom kategorikal -> {len(onehot_cols)} kolom one-hot:')
print(onehot_cols)

probe = train_set.head(1).copy()
probe['room_category'] = 'Type Z'
enc_unknown = enc.transform_frame(probe)
print('room_category=Type Z ->', {c: float(enc_unknown[c].iloc[0]) for c in enc_unknown if c.startswith('room_category=')})

### Dimensionality Reduction

This part covers fewer dimensions that keep the meaning of the data.

In [ ]:
# Write your code here

## Class Balance

This part covers the proportion of each class in the training data.

In [ ]:
counts = train_set[TARGET].value_counts().sort_index()
display(pd.DataFrame({'jumlah': counts, 'proporsi': (counts / counts.sum()).round(3)}))

_demo = Preprocessor(**DEFAULT_PREP_PARAMS).fit(train_set)
_X, _y = _demo.transform(train_set), train_set[TARGET].to_numpy()
resamplers = {'tanpa resampling': None, 'random oversampling': random_oversample,
              'SMOTE': smote, 'random undersampling': random_undersample}
rows = []
for name, fn in resamplers.items():
    yy = _y if fn is None else fn(_X, _y, seed=CFG.SEED)[1]
    rows.append({'metode': name, 'kelas 0': int((yy == 0).sum()), 'kelas 1': int((yy == 1).sum())})
display(pd.DataFrame(rows).set_index('metode'))

## Format Data

Formatting produces the structure the models read, such as a numeric matrix.

In [ ]:
y_train = train_set[TARGET].to_numpy()
y_val = val_set[TARGET].to_numpy()
print('y_train:', y_train.shape, y_train.dtype, ' y_val:', y_val.shape, y_val.dtype)

---

# Preparation Pipeline <a name="6"></a>

---

<!-- rubric v3: fitted steps, fitted_prep_steps -->

The pipeline combines the preparation steps so they run in one consistent order.

In [ ]:
print('Konfigurasi preprocessing:'); display(pd.Series(DEFAULT_PREP_PARAMS, name='nilai').to_frame())

data = prepare_datasets(train_df, test_df, val_size=VAL_SIZE, seed=CFG.SEED)
assert data['train_set'].equals(train_set), 'split/dedup di pipeline harus sama dengan langkah manual di atas'

for name in ['X_train', 'X_val', 'X_test']:
    X = data[name]
    print(f'{name}: shape={X.shape}, dtype={X.dtype}, NaN={int(np.isnan(X).sum())}')
print(f"y_train: {data['y_train'].shape}, y_val: {data['y_val'].shape}")
print(f"{len(data['prep'].feature_names_)} fitur akhir:")
print(data['prep'].feature_names_)

data['prep'].save(CFG.OUTPUT_DIR / 'preprocessor.json')
reloaded = Preprocessor.load(CFG.OUTPUT_DIR / 'preprocessor.json')
assert np.array_equal(reloaded.transform(val_set), data['X_val']), 'hasil load harus identik'
print('Preprocessor tersimpan di output/preprocessor.json dan hasil load identik.')

# variabel yang dipakai bagian pemodelan: X_train, y_train, X_val, y_val, X_test, prep
X_train, X_val, X_test = data['X_train'], data['X_val'], data['X_test']
prep = data['prep']

---

# Modeling and Validation <a name="7"></a>

---

The target is `is_canceled`. The models here are implemented from scratch, with math libraries such as NumPy.

## K-Nearest Neighbors

<!-- rubric v3: pairwise distances, val_neighbor_dists -->

This part covers the nearest neighbor classifier and its validation.

In [ ]:
# Write your code here

## Decision Tree Learning

This part covers one chosen variant, which is ID3, C4.5, or CART, and its validation.

In [ ]:
# Write your code here

## Logistic Regression

This part covers the model with L1, L2, and Elastic Net regularization.

In [ ]:
# Write your code here
lr_configs = {"LR (no reg)": dict(penalty=None), "LR (L1)": dict(penalty="l1", lam=0.001), "LR (L2)": dict(penalty="l2", lam=0.001), "LR (Elastic Net)": dict(penalty="elasticnet", lam=0.001, l1_ratio=0.5)}
lr_models = {}
for name, cfg in lr_configs.items():
    model = LogisticRegression(lr=0.01, n_iter=30, random_state=CFG.SEED, **cfg).fit(X_train, y_train)
    lr_models[name] = model
    print(f"{name:18s} iter={model.n_iter_:4d} loss={model.loss_history[-1]:.4f} zero_w={np.sum(model.coef_ == 0):3d} val_acc={model.score(X_val, y_val):.4f}")

plt.figure(figsize=(7, 4))
for name, model in lr_models.items():
    plt.plot(model.loss_history, label=name)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Logistic Regression Training Loss")
plt.legend()
plt.savefig(CFG.OUTPUT_DIR / "lr_loss.png")
plt.show()

## Support Vector Machine

<!-- rubric v3: total slack, slack_total -->

This part covers the linear, polynomial, and Gaussian kernels.

In [ ]:
# Write your code here

## Naive Bayes

This part covers the Naive Bayes classifier and its validation.

In [ ]:
# Write your code here

## Model Evaluation

<!-- figure palette: #6d2fa3 -->

This part compares the validation results of all models.

In [ ]:
# Write your code here
def to_binary(y):
    return (np.asarray(y).ravel() > 0).astype(int)

def confusion(y_true, y_pred):
    y_true, y_pred = to_binary(y_true), to_binary(y_pred)
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    return tp, tn, fp, fn

def roc_auc(y_true, scores):
    y_true = to_binary(y_true)
    ranks = pd.Series(np.asarray(scores, dtype=float)).rank().values
    n1, n0 = np.sum(y_true == 1), np.sum(y_true == 0)
    if n1 == 0 or n0 == 0:
        return np.nan
    return (np.sum(ranks[y_true == 1]) - n1 * (n1 + 1) / 2) / (n1 * n0)

def compute_metrics(y_true, y_pred, scores=None, fp_cost=2.0, fn_cost=1.0):
    tp, tn, fp, fn = confusion(y_true, y_pred)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    precision0 = tn / (tn + fn) if tn + fn else 0.0
    recall0 = tn / (tn + fp) if tn + fp else 0.0
    f1_0 = 2 * precision0 * recall0 / (precision0 + recall0) if precision0 + recall0 else 0.0
    return {"Accuracy": (tp + tn) / (tp + tn + fp + fn), "Precision": precision, "Recall": recall, "F1": f1, "Macro F1": (f1 + f1_0) / 2, "ROC AUC": roc_auc(y_true, scores) if scores is not None else np.nan, "TP": tp, "TN": tn, "FP": fp, "FN": fn, "Cost (2FP+FN)": fp_cost * fp + fn_cost * fn}

def get_scores(model, X):
    if hasattr(model, "predict_probability"):
        return np.asarray(model.predict_probability(X))[:, 1]
    if hasattr(model, "predict_proba"):
        return np.asarray(model.predict_proba(X))[:, 1]
    if hasattr(model, "decision_function"):
        return np.asarray(model.decision_function(X))
    return None

def evaluate_models(models, X, y):
    rows = {name: compute_metrics(y, model.predict(X), get_scores(model, X)) for name, model in models.items()}
    return pd.DataFrame(rows).T.sort_values("F1", ascending=False)

trained_models = {}
trained_models.update(lr_models)
eval_df = evaluate_models(trained_models, X_val, y_val)
display(eval_df.round(4))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
eval_df[["Accuracy", "Precision", "Recall", "F1", "ROC AUC"]].plot.bar(ax=axes[0], rot=30)
axes[0].set_title("Validation Metrics")
axes[0].set_ylim(0, 1)
best_name = eval_df.index[0]
tp, tn, fp, fn = confusion(y_val, trained_models[best_name].predict(X_val))
axes[1].imshow([[tn, fp], [fn, tp]], cmap="Blues")
for (i, j), v in np.ndenumerate(np.array([[tn, fp], [fn, tp]])):
    axes[1].text(j, i, v, ha="center", va="center")
axes[1].set_xticks([0, 1], ["Pred 0", "Pred 1"])
axes[1].set_yticks([0, 1], ["True 0", "True 1"])
axes[1].set_title(f"Confusion Matrix: {best_name}")
plt.tight_layout()
plt.savefig(CFG.OUTPUT_DIR / "model_evaluation.png")
plt.show()

def stratified_kfold_indices(y, k=5, seed=42):
    y = to_binary(y)
    rng = np.random.default_rng(seed)
    folds = [[] for _ in range(k)]
    for cls in (0, 1):
        idx = rng.permutation(np.where(y == cls)[0])
        for f, part in enumerate(np.array_split(idx, k)):
            folds[f].extend(part.tolist())
    return [np.array(sorted(f)) for f in folds]

def cross_validate(make_model, X, y, k=5, seed=42):
    X, y = np.asarray(X), to_binary(y)
    folds = stratified_kfold_indices(y, k, seed)
    rows = []
    for f in range(k):
        val_idx = folds[f]
        train_idx = np.concatenate([folds[g] for g in range(k) if g != f])
        model = make_model().fit(X[train_idx], y[train_idx])
        rows.append(compute_metrics(y[val_idx], model.predict(X[val_idx]), get_scores(model, X[val_idx])))
    df = pd.DataFrame(rows)
    return df.mean(), df.std()

def cross_validate_models(factories, X, y, k=5, seed=42):
    cols = ["Accuracy", "Precision", "Recall", "F1", "Macro F1", "ROC AUC", "Cost (2FP+FN)"]
    rows = {}
    for name, make_model in factories.items():
        mean, std = cross_validate(make_model, X, y, k, seed)
        rows[name] = {c: f"{mean[c]:.4f} ± {std[c]:.4f}" for c in cols}
    return pd.DataFrame(rows).T

lr_factories = {name: (lambda cfg=cfg: LogisticRegression(lr=0.01, n_iter=30, random_state=CFG.SEED, **cfg)) for name, cfg in lr_configs.items()}
cv_df = cross_validate_models(lr_factories, X_train, y_train, k=5, seed=CFG.SEED)
display(cv_df)

## Improvements (Optional)

This part covers hyperparameter tuning, cross-validation, and further visualization.

In [ ]:
# Write your code here
cs_models = {"LR (L2)": lr_models["LR (L2)"], "LR (L2, weighted)": LogisticRegression(penalty="l2", lam=0.001, class_weight={0: 2.0, 1: 1.0}, random_state=CFG.SEED).fit(X_train, y_train)}
val_probability = lr_models["LR (L2)"].predict_probability(X_val)[:, 1]
thresholds = np.linspace(0.05, 0.95, 91)
costs = [compute_metrics(y_val, (val_probability >= t).astype(int))["Cost (2FP+FN)"] for t in thresholds]
best_threshold = thresholds[int(np.argmin(costs))]
print(f"Best threshold = {best_threshold:.2f}  (theoretical 2/3 = {2/3:.2f})")
rows = {name: compute_metrics(y_val, m.predict(X_val), get_scores(m, X_val)) for name, m in cs_models.items()}
rows[f"LR (L2, thr={best_threshold:.2f})"] = compute_metrics(y_val, lr_models["LR (L2)"].predict(X_val, threshold=best_threshold), val_probability)
display(pd.DataFrame(rows).T[["Accuracy", "Precision", "Recall", "F1", "FP", "FN", "Cost (2FP+FN)"]].round(4))

In [ ]:
feature_idx = int(np.argmax(np.abs(lr_models["LR (L2)"].coef_)))
x1 = X_train[:, [feature_idx]]
viz_model = LogisticRegression(penalty=None, lr=0.01, n_iter=30, tol=0, record_history=True, random_state=CFG.SEED).fit(x1, y_train)
path = np.array(viz_model.theta_history)
t0 = np.linspace(path[:, 0].min() - 1.5, path[:, 0].max() + 1.5, 120)
t1 = np.linspace(path[:, 1].min() - 1.5, path[:, 1].max() + 1.5, 120)
T0, T1 = np.meshgrid(t0, t1)
Z = np.zeros_like(T0)
yv = to_binary(y_train)
for i in range(T0.shape[0]):
    P = np.clip(1 / (1 + np.exp(-(T0[i][:, None] + T1[i][:, None] * x1.ravel()[None, :]))), 1e-12, 1 - 1e-12)
    Z[i] = -np.mean(yv * np.log(P) + (1 - yv) * np.log(1 - P), axis=1)
fig, ax = plt.subplots(figsize=(7, 6))
cs = ax.contour(T0, T1, Z, levels=30, cmap="viridis")
ax.clabel(cs, inline=True, fontsize=7)
line, = ax.plot([], [], "r.-")
title = ax.set_title("")
ax.set_xlabel(r"$\theta_0$")
ax.set_ylabel(r"$\theta_1$")

def update(k):
    line.set_data(path[:k + 1, 0], path[:k + 1, 1])
    title.set_text(f"Epoch {k}  loss={viz_model.loss_history[k]:.4f}")
    return line, title

anim = FuncAnimation(fig, update, frames=len(path), interval=150, blit=True)
try:
    anim.save(CFG.OUTPUT_DIR / "lr_loss_contour.mp4", writer=FFMpegWriter(fps=8))
except Exception:
    anim.save(CFG.OUTPUT_DIR / "lr_loss_contour.gif", writer=PillowWriter(fps=8))
plt.close(fig)

---

# Comparison with scikit-learn <a name="8"></a>

---

This section compares each from-scratch model with its `scikit-learn` counterpart.

In [ ]:
# Write your code here
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    n = X_train.shape[0]
    sk_lr_configs = {"LR (no reg)": dict(C=np.inf, l1_ratio=0.0), "LR (L1)": dict(C=1 / (n * 0.001), l1_ratio=1.0), "LR (L2)": dict(C=1 / (n * 0.001), l1_ratio=0.0), "LR (Elastic Net)": dict(C=1 / (n * 0.001), l1_ratio=0.5)}
    sk_lr_models = {name: SkLogisticRegression(solver="saga", max_iter=5000, random_state=CFG.SEED, **cfg).fit(X_train, y_train) for name, cfg in sk_lr_configs.items()}
rows = []
for name in lr_models:
    ours = compute_metrics(y_val, lr_models[name].predict(X_val), get_scores(lr_models[name], X_val))
    sk = compute_metrics(y_val, sk_lr_models[name].predict(X_val), get_scores(sk_lr_models[name], X_val))
    agree = np.mean(lr_models[name].predict(X_val) == sk_lr_models[name].predict(X_val))
    coef_diff = np.max(np.abs(lr_models[name].coef_ - sk_lr_models[name].coef_.ravel()))
    rows.append({"Model": name, "Scratch F1": ours["F1"], "Sklearn F1": sk["F1"], "Scratch Acc": ours["Accuracy"], "Sklearn Acc": sk["Accuracy"], "Prediction Agreement": agree, "Max |coef diff|": coef_diff})
lr_compare_df = pd.DataFrame(rows).set_index("Model")
display(lr_compare_df.round(4))

---

# Saving and Loading Models <a name="9"></a>

---

<!-- artifacts: <model>_v3.pkl -->

This section stores the trained models and restores them for prediction.

In [ ]:
# Write your code here
for name, model in lr_models.items():
    path = CFG.OUTPUT_DIR / f"{name.replace(' ', '_').replace('(', '').replace(')', '').lower()}.pkl"
    model.save(path)
    loaded = LogisticRegression.load(path)
    print(f"{path.name}: identical predictions = {np.array_equal(model.predict(X_val), loaded.predict(X_val))}")

---

# Kaggle Submission <a name="10"></a>

---

The final pipeline and model are fitted on the original training set, and the test set is only transformed. The submission file contains the `id` and `is_canceled` columns.

In [ ]:
# Write your code here

---

# Error Analysis <a name="11"></a>

---

The analysis supports each step taken in this notebook.